In [1]:
import os
import GEOparse
import pandas as pd


# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {"sample_title": sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples' 
print("Initial metadata shape:", metadata_df.shape)

# Keep only samples with complete BCR survival endpoint information.
survival_cols = [
    'time to_bcr_(days)',
    'biochemical recurrence_within_5_years_of_treatment',
]
# metadata_df = metadata_df[
#     metadata_df[survival_cols].notna().all(axis=1)
#     & metadata_df[survival_cols].ne('NA').all(axis=1)
# ]

print("After filtering:       ", metadata_df.shape)

metadata_df.head()

Initial metadata shape: (95, 11)
After filtering:        (95, 11)


,sample_title,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,biochemical recurrence_within_5_years_of_treatment,time to_bcr_(days),gleason grade_group_at_rp,clinical stage_(ct)
samples,,,,,,,,,,,
GSM6674774,3T,Prostate tumor - FFPE,Radical prostatectomy,No,64,26.49,high,No,NA,G2_Gl7,cT_II
GSM6674775,5T,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,Yes,119,G2_Gl7,cT_II
GSM6674776,6T,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,Yes,1577,G2_Gl7,cT_I
GSM6674777,8T,Prostate tumor - FFPE,Radical prostatectomy,No,65,28.65,high,No,NA,G3_Gl7,cT_II
GSM6674778,16T,Prostate tumor - FFPE,Radical prostatectomy,No,59,21.87,high,No,NA,G4_5_higherGl7,cT_II


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')


df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_5543/1012403528.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,biochemical recurrence_within_5_years_of_treatment,time to_bcr_(days),gleason grade_group_at_rp,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM6674774,3T,Prostate tumor - FFPE,Radical prostatectomy,No,64,26.49,high,No,NA,G2_Gl7,...,1956,4,4,2,1369,185,44,1035,2,12
GSM6674775,5T,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,Yes,119,G2_Gl7,...,7408,23,23,16,3819,568,130,3070,21,37
GSM6674776,6T,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,Yes,1577,G2_Gl7,...,7664,25,18,11,3775,510,104,4391,7,30
GSM6674777,8T,Prostate tumor - FFPE,Radical prostatectomy,No,65,28.65,high,No,NA,G3_Gl7,...,3144,5,3,3,1777,229,40,1576,3,15
GSM6674778,16T,Prostate tumor - FFPE,Radical prostatectomy,No,59,21.87,high,No,NA,G4_5_higherGl7,...,10992,110,116,74,8678,1507,494,4905,14,367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM6674864,202T,Prostate tumor - FFPE,Radical prostatectomy,No,67,30.30,NA,Yes,7.9,G1_lessGl7,...,31,0,0,1,35,27,1,17,0,1
GSM6674865,245T,Prostate tumor - FFPE,Radical prostatectomy,No,50,25.43,NA,No,NA,G2_Gl7,...,1567,75,100,61,1709,294,87,1108,5,32
GSM6674866,253T,Prostate tumor - FFPE,Radical prostatectomy,No,57,27.43,NA,Yes,25.47,G3_Gl7,...,154,1,2,1,87,13,3,48,0,0


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data

In [4]:
metadata_df = metadata_df.reset_index()
metadata_df = metadata_df.set_index('sample_title')


df = pd.read_csv("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE216490&format=file&file=GSE216490%5FCountsTPC%5FERGstatus%2Etxt%2Egz", compression='gzip', sep='\t')
df = df.set_index('Genes')
df= df.T
df.index.name = "samples"
df.columns.name = None


df_combined = metadata_df.join(df, how='inner')
df_combined.reset_index()
# INFO: Uncomment to save the data to disk
# df_combined.to_csv(f'{GEO_ID}.csv', index=False)
df_combined


,samples,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,biochemical recurrence_within_5_years_of_treatment,time to_bcr_(days),gleason grade_group_at_rp,...,ENSG00000278588_HIST1H2BI,ENSG00000278615_C11orf98,ENSG00000278619_MRM1,ENSG00000278637_HIST1H4A,ENSG00000278662_GOLGA6L10,ENSG00000278677_HIST1H2AM,ENSG00000278705_HIST1H4B,ENSG00000278828_HIST1H3H,ENSG00000278845_MRPL45,ENSG00000280165_PCDH20
3T,GSM6674774,Prostate tumor - FFPE,Radical prostatectomy,No,64,26.49,high,No,NA,G2_Gl7,...,0.00,2.16,1.00,0.0,11.82,2.01,4.0,2.00,8.00,1.00
5T,GSM6674775,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,Yes,119,G2_Gl7,...,0.00,14.97,1.00,1.0,0.02,0.00,3.0,1.00,16.70,1.00
6T,GSM6674776,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,Yes,1577,G2_Gl7,...,0.00,9.11,2.00,0.0,0.00,1.02,5.0,3.00,13.31,0.00
8T,GSM6674777,Prostate tumor - FFPE,Radical prostatectomy,No,65,28.65,high,No,NA,G3_Gl7,...,0.00,6.40,2.00,1.0,3.91,2.10,5.0,2.00,18.27,10.00
16T,GSM6674778,Prostate tumor - FFPE,Radical prostatectomy,No,59,21.87,high,No,NA,G4_5_higherGl7,...,0.00,16.45,12.00,6.0,1.72,2.01,6.0,28.00,44.22,4.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282T,GSM6674847,Prostate tumor - FFPE,Radical prostatectomy,No,44,27.02,high,No,NA,G1_lessGl7,...,0.00,36.30,11.99,2.0,57.19,5.03,15.0,51.00,85.51,6.00
285T,GSM6674848,Prostate tumor - FFPE,Radical prostatectomy,No,61,25.34,high,Yes,254,G2_Gl7,...,1.00,18.33,19.00,4.0,17.90,4.01,20.0,16.00,43.27,5.00
287T,GSM6674849,Prostate tumor - FFPE,Radical prostatectomy,No,67,31.57,low,Yes,1652,G2_Gl7,...,1.00,33.87,12.00,5.0,17.39,9.00,8.0,38.00,51.35,13.00
288T,GSM6674850,Prostate tumor - FFPE,Radical prostatectomy,No,67,26.37,low,NA,NA,G4_5_higherGl7,...,4.49,147.03,50.94,11.0,160.07,48.09,71.0,61.04,388.14,80.97


In [5]:
df_combined[df_combined['time to_bcr_(days)'] != 'NA']

,samples,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,biochemical recurrence_within_5_years_of_treatment,time to_bcr_(days),gleason grade_group_at_rp,...,ENSG00000278588_HIST1H2BI,ENSG00000278615_C11orf98,ENSG00000278619_MRM1,ENSG00000278637_HIST1H4A,ENSG00000278662_GOLGA6L10,ENSG00000278677_HIST1H2AM,ENSG00000278705_HIST1H4B,ENSG00000278828_HIST1H3H,ENSG00000278845_MRPL45,ENSG00000280165_PCDH20
5T,GSM6674775,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,Yes,119,G2_Gl7,...,0.00,14.97,1.00,1.0,0.02,0.00,3.00,1.00,16.70,1.0
6T,GSM6674776,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,Yes,1577,G2_Gl7,...,0.00,9.11,2.00,0.0,0.00,1.02,5.00,3.00,13.31,0.0
28T,GSM6674779,Prostate tumor - FFPE,Radical prostatectomy,No,73,23.94,high,Yes,158,G2_Gl7,...,0.00,2.31,4.00,0.0,5.98,2.00,1.00,3.00,19.56,2.0
33T,GSM6674780,Prostate tumor - FFPE,Radical prostatectomy,No,63,27.01,high,Yes,788,G1_lessGl7,...,0.00,15.34,8.00,1.0,10.03,1.12,3.92,1.00,30.33,3.0
35T,GSM6674781,Prostate tumor - FFPE,Radical prostatectomy,No,65,23.18,low,Yes,125,G3_Gl7,...,0.00,17.91,2.00,0.0,4.72,7.27,17.00,6.00,61.13,0.0
37T,GSM6674782,Prostate tumor - FFPE,Radical prostatectomy,No,61,27.31,low,Yes,852,G4_5_higherGl7,...,4.50,16.24,6.00,3.0,21.41,6.04,11.95,12.00,51.67,2.0
53T,GSM6674790,Prostate tumor - FFPE,Radical prostatectomy,No,69,26.61,low,Yes,1377,G2_Gl7,...,0.00,3.30,5.00,2.0,2.41,1.02,9.00,6.00,27.31,6.0
55T,GSM6674792,Prostate tumor - FFPE,Radical prostatectomy,No,64,27.06,low,Yes,1484,G2_Gl7,...,0.96,3.95,2.00,0.0,23.21,0.00,10.03,1.00,24.32,3.0
66T,GSM6674799,Prostate tumor - FFPE,Radical prostatectomy,No,63,27.18,low,Yes,947,G3_Gl7,...,0.00,3.42,4.00,0.0,1.36,1.00,3.00,2.00,8.24,0.0
70T,GSM6674801,Prostate tumor - FFPE,Radical prostatectomy,No,62,24.91,high,Yes,92,G4_5_higherGl7,...,0.00,15.90,3.00,0.0,8.55,0.00,1.00,2.00,13.42,0.0
